# 02 Inspect tiles and derived labels
Draw the tile grid and the boxes on one image, and colour each tile by its derived label. Use this to choose the tile size: plants should span roughly one tile.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mp
from exw.config import load_config
from exw.constants import EXW, OTHER, HEALTHY, IGNORE
from exw.data.dataset import load_image
from exw.data.labels import tile_labels
from exw.data.manifest import load_annotations, load_manifest
from exw.data.tiling import extract_tiles

cfg = load_config('configs/default.yaml')
tc = cfg['tiles']
df = load_manifest(cfg['paths']['data_dir']); ann = load_annotations(cfg['paths']['data_dir'])
row = df[df.has_boxes & (df.farm_label == 'exw_suspect')].iloc[0]
img = load_image(cfg['paths']['data_dir'], row['path'])
tiles, coords, grid = extract_tiles(img, tc['size'], tc['stride'])
boxes = ann[row['image_id']]['boxes']
lab = tile_labels(coords, boxes, tc['size'], tc['box_frac'], tc['ignore_frac'])
colour = {EXW: 'red', OTHER: 'orange', HEALTHY: None, IGNORE: 'grey'}
fig, ax = plt.subplots(figsize=(11, 8)); ax.imshow(img)
for (x, y), l in zip(coords, lab):
    if colour[int(l)]: ax.add_patch(mp.Rectangle((x, y), tc['size'], tc['size'], fill=True, alpha=0.18, color=colour[int(l)]))
for b in boxes: ax.add_patch(mp.Rectangle((b['x0'], b['y0']), b['x1']-b['x0'], b['y1']-b['y0'], fill=False, edgecolor='cyan', lw=1.5))
ax.set_title(f"{row['image_id']}  tiles={len(coords)}  red=EXW orange=other grey=ignored cyan=boxes"); ax.axis('off')

In [ ]:
# how many tiles of each label, and what fraction of boxes are too small to fill a tile?
import numpy as np
print({int(k): int((lab == k).sum()) for k in np.unique(lab)})
small = [b for b in boxes if (b['x1']-b['x0'])*(b['y1']-b['y0']) < 0.5 * tc['size'] ** 2]
print(f'{len(small)} of {len(boxes)} boxes are smaller than half a tile -> consider a smaller tile or higher resolution')